In [1]:
import pandas as pd
import fastembed 
import qdrant_client

# FASE 1 INDICIZZAZIONE

In [2]:
import qdrant_client
from qdrant_client import models
client = qdrant_client.QdrantClient('http://localhost:6333', timeout=1000)

In [3]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
#import SentenceTransfomers
dense_embedding_model = TextEmbedding("jinaai/jina-embeddings-v3", cache_dir = './fastembed/')
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25", cache_dir = './fastembed/')
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0", cache_dir = './fastembed/')

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


config.json: 0.00B [00:00, ?B/s]

C:\Users\andre\PycharmProjects\corso_ai\.venv2\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\PycharmProjects\corso_ai\modulo_6_rag\fastembed\models--jinaai--jina-embeddings-v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

C:\Users\andre\PycharmProjects\corso_ai\.venv2\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\PycharmProjects\corso_ai\modulo_6_rag\fastembed\models--colbert-ir--colbertv2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.onnx:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [4]:
client.create_collection(
            collection_name='rag_capitoli',
            vectors_config={
                "dense": models.VectorParams(
                    size=1024,
                    distance=models.Distance.COSINE
                ),
                 "colbert": models.VectorParams(
                size=128,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                ),
                hnsw_config=models.HnswConfigDiff(m=0)  # Disable HNSW for reranking
        )
                
            },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [7]:
df = pd.read_excel('../../data/INDICI_PULITI_MVP_01.xlsx', sheet_name=0)

In [8]:
df.head()

,id_indice,disciplina,ordine_scuola,liv_min,liv_max,unita,lezione,sotto_lezione,descrizione_breve,tag,is_uploaded
0,SS2-3-FIL-001,Filosofia,Scuola Secondaria di Secondo Grado,10,10,LA CIVILTÀ GRECA E LA FILOSOFIA,CHE COS’È LA FILOSOFIA?,Il significato del termine filosofia,Si esplora l'etimologia e il significato profo...,"Filosofia, Significato, Etimologia, Sapienza, ...",True
1,SS2-3-FIL-002,Filosofia,Scuola Secondaria di Secondo Grado,10,10,LA CIVILTÀ GRECA E LA FILOSOFIA,CHE COS’È LA FILOSOFIA?,La ricerca della verità,Viene esaminato il ruolo centrale della ricerc...,"Verità, Ricerca, Filosofia, Ragione, Realtà, C...",True
2,SS2-3-FIL-003,Filosofia,Scuola Secondaria di Secondo Grado,10,10,LA CIVILTÀ GRECA E LA FILOSOFIA,CHE COS’È LA FILOSOFIA?,Lo stupore di esistere,Si analizza lo stupore come punto di partenza ...,"Stupore, Esistenza, Meraviglia, Filosofia, Ori...",True
3,SS2-3-FIL-004,Filosofia,Scuola Secondaria di Secondo Grado,10,10,LA CIVILTÀ GRECA E LA FILOSOFIA,CHE COS’È LA FILOSOFIA?,La ragione illumina la mente,Si approfondisce il ruolo della ragione come s...,"Ragione, Mente, Illuminazione, Filosofia, Razi...",True
4,SS2-3-FIL-005,Filosofia,Scuola Secondaria di Secondo Grado,10,10,LA CIVILTÀ GRECA E LA FILOSOFIA,CHE COS’È LA FILOSOFIA?,Quali sono le domande che si pone la filosofia,Vengono esplorate le domande fondamentali che ...,"Domande filosofiche, Esistenza, Conoscenza, Et...",True


In [9]:
id_indices = list(df.id_indice)

In [10]:
len(id_indices)

1220

In [11]:
disciplina = 'filosofia'
ordine_scuola = list(df.ordine_scuola)
liv_min = list(df.liv_min)
liv_max = list(df.liv_max)
unita = list(df.unita)
lezione = list(df.lezione)
titolo = [unit + '\n' + lez for unit, lez in zip(unita, lezione)]
sotto_lezione = [slez if type(slez) == str else '' for slez in df.sotto_lezione ]
descrizione_breve = list(df.descrizione_breve)
descrizione = [unit + '\n' + lez + '\n' + sotto_lez + '\n' + descr for unit, lez, sotto_lez, descr in zip(unita, lezione, sotto_lezione, descrizione_breve)]
tag = list(df.tag)

In [12]:
descrizione[0]

"LA CIVILTÀ GRECA E LA FILOSOFIA\nCHE COS’È LA FILOSOFIA?\nIl significato del termine filosofia\nSi esplora l'etimologia e il significato profondo del termine 'filosofia', intesa come amore per la sapienza. Si analizza come la filosofia si distingua da altre forme di conoscenza."

In [13]:
dense_embeddings = list(dense_embedding_model.embed(text for text in descrizione))

In [14]:
late_interaction_embeddings = list(late_interaction_embedding_model.embed(text for text in descrizione))

In [15]:
bm25_embeddings = list(bm25_embedding_model.embed(text for text in descrizione))

In [16]:
from qdrant_client.models import PointStruct

points = []
for indice,(idx, dense_embedding, bm25_embedding, late_interaction_embedding, tit, ord_scuola, liv_minimo, liv_maximo, descr, tg) in enumerate(zip(id_indices, dense_embeddings, bm25_embeddings, late_interaction_embeddings, titolo, ordine_scuola, liv_min, liv_max,  descrizione, tag)):
  
    point = PointStruct(
        id=indice,
        vector={
            "dense": dense_embedding,
            "colbert": late_interaction_embedding,
            "bm25": bm25_embedding.as_object(),
        },
        payload={"indice":idx, "descr": descr, "disciplina": disciplina, "ordine_scuola": ord_scuola, "liv_min": liv_minimo, "liv_max": liv_maximo, "tag": tg, "titolo": tit}
    )
    points.append(point)

In [17]:
for batch in range(len(points) // 20):
    operation_info = client.upsert(
    collection_name="rag_capitoli",
    points=points[batch * 20:(batch+1) * 20]
)
operation_info = client.upsert(
    collection_name="rag_capitoli",
    points=points[batch * 20:])
print(operation_info)

operation_id=62 status=<UpdateStatus.COMPLETED: 'completed'>


# FASE 2: FASE QUERY

In [18]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.eval()

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [19]:
import torch
torch.device('cuda' if torch.cuda.is_available() else 'cpu')


device(type='cuda')

In [20]:
model.to('cuda')

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [21]:
#query = "Si descrive Platone e il mondo delle idee"
lezione = "Il prof. Matteo Saudino, ideatore del canale YouTube <i>BarbaSophia</i>, legge un passaggio del Manifesto ed Engels in cui si desrivono il ruolo e la trasformazione della borghesia."

In [24]:
dense_vector = next(dense_embedding_model.query_embed(lezione))
sparse_vector = next(bm25_embedding_model.query_embed(lezione))
late_interaction_vector = next(late_interaction_embedding_model.query_embed(lezione))

In [26]:
prefetch = [
        models.Prefetch(
            query=dense_vector,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_vector.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

In [28]:
results = client.query_points(
         "rag_capitoli",
        prefetch=prefetch,
        query=late_interaction_vector,
        using="colbert",
        with_payload=True,
        limit=20,
)

points = results.points

ScoredPoint(id=421, version=22, score=36.361645, payload={'indice': 'SS2-4-FIL-086', 'descr': 'Le radici della scienza moderna\nGalileo Galilei\nIl secondo principio della dinamica\nAnalisi della scoperta di Galileo relativa al moto uniformemente accelerato, in particolare la legge secondo cui lo spazio percorso da un corpo in caduta libera è direttamente proporzionale al quadrato del tempo impiegato.', 'disciplina': 'filosofia', 'ordine_scuola': 'Scuola Secondaria di Secondo Grado', 'liv_min': 11, 'liv_max': 11, 'tag': 'Galileo Galilei, moto uniformemente accelerato, legge oraria del moto, caduta dei gravi, accelerazione di gravità, secondo principio della dinamica, leggi del moto, fisica, spazio e tempo, dinamica', 'titolo': 'Le radici della scienza moderna\nGalileo Galilei'}, vector=None, shard_key=None, order_value=None)

In [33]:
#model.to('cuda')
results = model.rerank(lezione, [point.payload['descr'] for point in points])

# Results are sorted by relevance score (highest first)
for result in results:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()

Score: 0.1886
Document: Marx e il materialismo storico
Il Manifesto del partito comunista

Presenta il "Manifesto del partit...

Score: 0.0782
Document: Marx e il materialismo storico
Economia borghese e alienazione
I quattro aspetti dell’alienazione
De...

Score: 0.0278
Document: Marx e il materialismo storico
Il Manifesto del partito comunista
Il socialismo scientifico
Spiega i...

Score: -0.0332
Document: Marx e il materialismo storico
Economia borghese e alienazione
Il confronto con Feuerbach
Approfondi...

Score: -0.0567
Document: Marx e il materialismo storico
Economia borghese e alienazione
Gli economisti classici
Analizza il r...

Score: -0.0689
Document: Marx e il materialismo storico
Storia e società
Il materialismo dialettico
Spiega il materialismo di...

Score: -0.1017
Document: Husserl e la fenomenologia
Edmund Husserl
Svolta trascendentale
Analisi del passaggio di Husserl da ...

Score: -0.1565
Document: Kant: individuo e società
Lo stato di diritto

Si definisce il conce

In [34]:
results = [point.payload for point in points if point.payload['descr'] in [result['document'] for result in results]]

In [37]:
SYSTEM_MESSAGE = """Sei un esperto di sistemi educativi. Il tuo compito è determinare il livello di istruzione necessario (scala 0-12) per una lezione basandoti su capitoli di riferimento recuperati.

REGOLE DI RAGIONAMENTO:
1. Logica del Vincolo (Max dei Minimi): Se una lezione tratta più argomenti, il livello MINIMO della lezione deve coincidere con il livello più ALTO tra i minimi dei singoli argomenti. (Esempio: Argomento A liv. 10 + Argomento B liv. 12 = Livello Minimo Lezione 12).
2. Filtro Pertinenza: Usa i dati dei capitoli solo se sono realmente coerenti con l'argomento della lezione. Se i capitoli recuperati sono fuori tema, ignora i loro numeri e stima il livello basandoti sulla complessità concettuale della descrizione.
3. Coerenza del Range: Se la lezione è focalizzata su un singolo tema specifico, il livello minimo e massimo dovrebbero coincidere.
4. Livelli target: 0-4 (Elementari), 5-7 (Medie), 8-12 (Superiori).
5. Gerarchia della Pertinenza: > * Considera "Pertinenti" solo i capitoli che trattano l'argomento centrale della descrizione (es. se la lezione è su Aristotele, i capitoli su Aristotele sono l'unico riferimento valido).

. I capitoli che trattano critiche successive o raccordi storici (es. Bacone che critica Aristotele, o Kant che cita Platone) devono essere considerati "Secondari" e i loro livelli non devono influenzare il calcolo del vincolo, a meno che quegli autori non siano esplicitamente menzionati nella descrizione della lezione.

REGOLE DI ESCLUSIONE CATEGORICA:
- Argomento Centrale: Il calcolo del livello deve basarsi solo su capitoli che condividono lo stesso ambito applicativo della descrizione. 
- Divieto di Approfondimento Forzato: Non alzare il livello minimo usando capitoli "difficili" solo perché trattano temi simili in modo più approfondito. Se la lezione parla di Costituzione, usa i livelli della Costituzione.
"""

In [38]:
PROMPT_TEMPLATE = """### INPUT
Descrizione Lezione: "{{descrizione_lezione}}"
Capitoli Recuperati: 
{{risultati_retriever}}

### ISTRUZIONI
1. Analizza la descrizione e identifica i temi principali.
2. Per ogni tema, verifica se esiste un capitolo recuperato pertinente.
3. Applica la "Logica del Vincolo": identifica il livello minimo più alto tra tutti i temi necessari per comprendere la lezione.
4. Se i capitoli non sono pertinenti, indica "stima_autonoma" e calcola il livello in base alla difficoltà dei concetti espressi.
### FORMATO OUTPUT (JSON)
{
  "livello_min_consigliato": int >= 0,
  "livello_max_consigliato": int <= 12,
  "motivazione": "Spiega quale tema ha determinato il livello minimo finale",
  "fonte_livello": "capitoli_recuperati" O "stima_autonoma"
}"""

In [41]:
df_true = pd.read_excel('../../data/V2_elenco-contenuti-editori_AGGIORNATI_da-backend.xlsx')

In [42]:
df_filo = df_true.loc[df_true.Discipline.str.lower().str.contains('filosofia')]

In [43]:
print(len(df_filo))
df_filo.head()

1469


,esId,Link,Editore,Titolo,Descrizione,Tipo,Discipline,Ordine di scuola min,Ordine di scuola max,LIV MIN = LIV MAX,...,Visualizzazioni,Vis. Docenti,Vis. Studenti,Utilizzi,Segnalibri,Like,type,id,data,_score
441,QIxPYZkBshGDi9jXBFuK,https://www.youtube.com/watch?v=ZwQEp4vD3KE,BarbaSophia,Platone e la scienza metretica,"Il prof. Matteo Saudino, ideatore del canale Y...",video,Filosofia,10.0,10.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
442,QYxPYZkBshGDi9jXBls6,https://www.youtube.com/watch?v=HzcixdIKCTA,BarbaSophia,Marx: la borghesia e il mercato globale,"Il prof. Matteo Saudino, ideatore del canale Y...",video,Filosofia,12.0,12.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
443,QoxPYZkBshGDi9jXB1uH,https://www.youtube.com/watch?v=qKsveWEW-Q0,BarbaSophia,Platone: una vita politica,"Il prof. Matteo Saudino, ideatore del canale Y...",video,Filosofia,10.0,10.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
444,RoxPYZkBshGDi9jXC1t8,https://www.youtube.com/watch?v=u-p9Uon6IkY,BarbaSophia,Cos parlò Zarathustra di Nietzsche: il saluto,"Il prof. Matteo Saudino, ideatore del canale Y...",video,Filosofia,12.0,12.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
445,Q4xPYZkBshGDi9jXCFu0,https://www.youtube.com/watch?v=IYCIx3zjDsk,BarbaSophia,Pico della Mirandola: la dignità dell’uomo,"Il prof. Matteo Saudino, ideatore del canale Y...",video,Filosofia,11.0,11.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [45]:
df_filo.Descrizione.iloc[0]

'Il prof. Matteo Saudino, ideatore del canale YouTube <i>BarbaSophia</i>, in questa lezione spiega tre dialoghi giovanili di Platone in cui si affrontano temi che il filosofo affronterà anche nella sua maturità: il linguaggio, la virtù e la retorica'

In [47]:
import os

In [53]:
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')

In [54]:
from google import genai
from google.genai import types
import os
import json
from typing import List

In [55]:
GEMINI_MODEL = 'gemini-2.5-flash'
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

In [57]:
from pydantic import BaseModel, Field
from typing import Literal

class LivelloResult(BaseModel):
    livello_min_consigliato: int 
    livello_max_consigliato: int
    motivazione: str = Field(description = "Breve spiegazione del perché questi livelli sono stati scelti in base ai capitoli trovati")
    fonte_livello: Literal["capitoli_recuperati","stima_autonoma_su_descrizione"]


In [60]:
def assign_min_max(query):
    dense_vector = next(dense_embedding_model.embed(query)) #next(dense_embedding_model.query_embed(query))
    sparse_vector = next(bm25_embedding_model.embed(query))#next(bm25_embedding_model.query_embed(query))
    late_interaction_vector = next(late_interaction_embedding_model.embed(query)) #next(late_interaction_embedding_model.query_embed(query))
    prefetch = [
            models.Prefetch(
                query=dense_vector,
                using="dense",
                limit=20,
            ),
            models.Prefetch(
                query=models.SparseVector(**sparse_vector.as_object()),
                using="bm25",
                limit=1,
            ),
        ]
    results = client.query_points(
             "rag_capitoli",
            prefetch=prefetch,
            query=late_interaction_vector,
            using="colbert",
            with_payload=True,
            limit=20,
    )
    
    points = results.points
    results = model.rerank(query, [point.payload['descr'] for point in points])
    results = [point.payload for point in points if point.payload['descr'] in [result['document'] for result in results[:10]]]
    
    prompt = PROMPT_TEMPLATE.replace('{{descrizione_lezione}}', query).replace('{{risultati_retriever}}', '\n'.join([f'{index}. Descrizione: {res["descr"]}\nTitolo: {res["titolo"]}\nLivello Minimo: {res["liv_min"]}\nLivello Massimo: {res["liv_max"]}' for index, res in enumerate(results)]))
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=content_list,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_MESSAGE,
            temperature=0.01,
            top_k=1,
            response_mime_type='application/json',
            response_schema=LivelloResult)
    )

    return result.parsed


In [61]:
results_min = 0
results_max = 0

for idx in range(0,100):
    print('Descrizione: ' + str(df_filo['Descrizione'].iloc[idx]))
    print('True level min: ' + str(df_filo['Ordine di scuola min'].iloc[idx]))
    print('True level max: ' + str(df_filo['Ordine di scuola max'].iloc[idx]))
    print('-'*100)
    res = assign_min_max(df_filo.Descrizione.iloc[idx])
    print('Predicted level min: ' + str(res.livello_min_consigliato))
    print('Predicted level max: ' + str(res.livello_max_consigliato))
    print('-'*100)

    print('Motivazione: ' + res.motivazione)
    print('fonte_livello: '+ res.fonte_livello)
    print('-'*100)
    results_min += 1 if (df_filo['Ordine di scuola min'].iloc[idx] == res.livello_min_consigliato) else 0
    results_max += 1 if df_filo['Ordine di scuola max'].iloc[idx] == res.livello_max_consigliato else 0




Descrizione: Il prof. Matteo Saudino, ideatore del canale YouTube <i>BarbaSophia</i>, in questa lezione spiega tre dialoghi giovanili di Platone in cui si affrontano temi che il filosofo affronterà anche nella sua maturità: il linguaggio, la virtù e la retorica
True level min: 10.0
True level max: 10.0
----------------------------------------------------------------------------------------------------
Predicted level min: 10
Predicted level max: 10
----------------------------------------------------------------------------------------------------
Motivazione: I capitoli recuperati non sono sufficientemente pertinenti all'argomento specifico della lezione (tre dialoghi giovanili di Platone e i temi di linguaggio, virtù e retorica in essi contenuti). Il livello è stato stimato autonomamente, considerando che l'analisi dei dialoghi giovanili di Platone e dei concetti filosofici fondamentali come linguaggio, virtù e retorica è tipicamente affrontata nel percorso di studi superiori (liceo)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [90]:
idx

99